# EDA: 気象データ × 処方枚数 相関分析

## 仮説
- **H1**: 悪天候（大雪・低温）は短期的に来局を減少させる
- **H2**: しかし定期薬は尽きるので、悪天候後2-3日でリバウンド来局増
- **H3**: 冬季は天候に関係なく来局するため、天候相関が弱まる
- **H4**: インフルエンザ定点報告は処方枚数の先行指標として使える

## データソース
| データ | ソース | 粒度 |
|--------|--------|------|
| 処方実績 | Musubi (sample) | 日次・店舗別 |
| 気象データ | 気象庁 JMA | 日次・観測所別 |
| 感染症報告 | NIID IDWR | 週次・都道府県別 |

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Django setup
sys.path.insert(0, os.path.join(os.getcwd(), 'backend'))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings')

import django
django.setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

# Japanese font support
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

from apps.analytics.models import InfluenzaReport, PrescriptionRecord, WeatherRecord, AREA_STATION_MAP
from apps.stores.models import Store

print('Setup complete')

## 1. データ読み込み

In [ ]:
# Prescription data
rx_qs = PrescriptionRecord.objects.select_related('store').values(
    'store__name', 'store__area', 'date', 'count'
)
rx_df = pd.DataFrame(list(rx_qs))
rx_df.rename(columns={'store__name': 'store', 'store__area': 'area', 'count': 'prescriptions'}, inplace=True)
rx_df['date'] = pd.to_datetime(rx_df['date'])

# Weather data
w_qs = WeatherRecord.objects.values(
    'station_name', 'date', 'avg_temperature', 'max_temperature', 'min_temperature',
    'precipitation', 'humidity', 'snowfall', 'snow_depth'
)
weather_df = pd.DataFrame(list(w_qs))
weather_df['date'] = pd.to_datetime(weather_df['date'])
for c in ['avg_temperature','max_temperature','min_temperature','precipitation','humidity','snowfall','snow_depth']:
    weather_df[c] = pd.to_numeric(weather_df[c], errors='coerce')

# Influenza data
flu_qs = InfluenzaReport.objects.filter(
    disease='インフルエンザ', prefecture='北海道'
).values('year', 'week', 'patients', 'total_reports')
flu_df = pd.DataFrame(list(flu_qs))
if not flu_df.empty:
    from datetime import date
    flu_df['date'] = flu_df.apply(
        lambda r: pd.Timestamp(date.fromisocalendar(int(r['year']), max(int(r['week']),1), 1)), axis=1
    )
    flu_df['patients'] = pd.to_numeric(flu_df['patients'], errors='coerce')

print(f'Prescription: {len(rx_df):,} rows ({rx_df["date"].min().date()} ~ {rx_df["date"].max().date()})')
print(f'Weather:      {len(weather_df):,} rows')
print(f'Influenza:    {len(flu_df):,} rows')
print(f'Areas:        {sorted(rx_df["area"].unique())}')
print(f'Stores:       {rx_df["store"].nunique()}')

## 2. 基礎統計

In [ ]:
# Area-level stats
area_stats = rx_df.groupby('area')['prescriptions'].agg(['mean','std','min','max','count'])
area_stats = area_stats.sort_values('mean', ascending=False)
area_stats.round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Day-of-week pattern
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
rx_df['dow'] = rx_df['date'].dt.dayofweek
dow_avg = rx_df.groupby('dow')['prescriptions'].mean()
axes[0].bar(range(7), dow_avg.values, color=['#4C72B0']*5 + ['#DD8452','#C44E52'])
axes[0].set_xticks(range(7))
axes[0].set_xticklabels(dow_names)
axes[0].set_title('Day-of-Week Pattern')
axes[0].set_ylabel('Avg Prescriptions')

# Monthly pattern
rx_df['month'] = rx_df['date'].dt.month
month_avg = rx_df.groupby('month')['prescriptions'].mean()
colors = ['#C44E52' if m in [1,2,12] else '#4C72B0' for m in range(1,13)]
axes[1].bar(range(1,13), month_avg.values, color=colors)
axes[1].set_xticks(range(1,13))
axes[1].set_xlabel('Month')
axes[1].set_title('Monthly Pattern (red = flu season)')
axes[1].set_ylabel('Avg Prescriptions')

# Area comparison
area_means = rx_df.groupby('area')['prescriptions'].mean().sort_values()
axes[2].barh(range(len(area_means)), area_means.values, color='#4C72B0')
axes[2].set_yticks(range(len(area_means)))
axes[2].set_yticklabels(area_means.index)
axes[2].set_title('Area Avg Prescriptions')
axes[2].set_xlabel('Avg Prescriptions/day')

plt.tight_layout()
plt.savefig('notebooks/fig_01_basic_patterns.png', bbox_inches='tight')
plt.show()

## 3. 気象データ概要

In [ ]:
asahikawa_w = weather_df[weather_df['station_name'] == 'Asahikawa'].copy() if 'Asahikawa' in weather_df['station_name'].values else weather_df[weather_df['station_name'] == list(weather_df['station_name'].unique())[0]].copy()
# Fallback: use whatever station we have
station = weather_df['station_name'].unique()[0] if len(weather_df) > 0 else 'N/A'
asahikawa_w = weather_df[weather_df['station_name'] == station].copy()
asahikawa_w = asahikawa_w.sort_values('date')

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

# Temperature
axes[0].fill_between(asahikawa_w['date'], asahikawa_w['min_temperature'], 
                     asahikawa_w['max_temperature'], alpha=0.3, color='#C44E52')
axes[0].plot(asahikawa_w['date'], asahikawa_w['avg_temperature'], color='#C44E52', linewidth=1)
axes[0].axhline(y=0, color='black', linewidth=0.5, linestyle='--')
axes[0].set_ylabel('Temperature (C)')
axes[0].set_title(f'{station} Weather Data')

# Precipitation
axes[1].bar(asahikawa_w['date'], asahikawa_w['precipitation'].fillna(0), 
            color='#4C72B0', width=1)
axes[1].set_ylabel('Precipitation (mm)')

# Snowfall & snow depth
axes[2].bar(asahikawa_w['date'], asahikawa_w['snowfall'].fillna(0), 
            color='#55A868', alpha=0.7, width=1, label='Snowfall')
ax2r = axes[2].twinx()
ax2r.plot(asahikawa_w['date'], asahikawa_w['snow_depth'].fillna(0), 
          color='#8172B2', linewidth=1.5, label='Snow depth')
axes[2].set_ylabel('Snowfall (cm)')
ax2r.set_ylabel('Snow depth (cm)')
axes[2].legend(loc='upper left')
ax2r.legend(loc='upper right')

# Humidity
axes[3].plot(asahikawa_w['date'], asahikawa_w['humidity'], color='#CCB974', linewidth=1)
axes[3].set_ylabel('Humidity (%)')
axes[3].set_xlabel('Date')

for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig('notebooks/fig_02_weather_overview.png', bbox_inches='tight')
plt.show()

## 4. 同時相関分析（当日天候 × 当日処方）

In [ ]:
# Merge: Asahikawa area prescriptions (daily avg) + Asahikawa weather
rx_asahikawa = rx_df[rx_df['area'] == 'Asahikawa'] if 'Asahikawa' in rx_df['area'].values else rx_df[rx_df['area'] == list(rx_df['area'].unique())[0]]
# Use the most common area
main_area = rx_df['area'].value_counts().index[0]
rx_main = rx_df[rx_df['area'] == main_area].copy()
rx_daily = rx_main.groupby('date')['prescriptions'].agg(['mean','sum','count']).reset_index()
rx_daily.rename(columns={'mean': 'prescriptions', 'sum': 'total_rx'}, inplace=True)

merged = pd.merge(rx_daily, asahikawa_w, on='date', how='inner')
print(f'Merged: {len(merged)} rows ({main_area} area x {station} station)')

In [ ]:
weather_vars = [
    ('avg_temperature', 'Avg Temp (C)'),
    ('precipitation', 'Precipitation (mm)'),
    ('humidity', 'Humidity (%)'),
    ('snowfall', 'Snowfall (cm)'),
    ('snow_depth', 'Snow Depth (cm)'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

results = []
for i, (col, label) in enumerate(weather_vars):
    valid = merged.dropna(subset=[col, 'prescriptions'])
    if len(valid) < 10:
        continue
    r, p = stats.pearsonr(valid[col], valid['prescriptions'])
    results.append({'Variable': label, 'r': r, 'p': p, 'n': len(valid)})
    
    ax = axes[i]
    ax.scatter(valid[col], valid['prescriptions'], alpha=0.3, s=15, color='#4C72B0')
    # Regression line
    z = np.polyfit(valid[col], valid['prescriptions'], 1)
    p_line = np.poly1d(z)
    x_range = np.linspace(valid[col].min(), valid[col].max(), 100)
    ax.plot(x_range, p_line(x_range), 'r-', linewidth=2)
    ax.set_xlabel(label)
    ax.set_ylabel('Prescriptions')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    ax.set_title(f'{label}\nr={r:.3f} {sig}')

# Hide unused subplot
axes[5].axis('off')

plt.suptitle('Same-Day Correlation: Weather vs Prescriptions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('notebooks/fig_03_same_day_correlation.png', bbox_inches='tight')
plt.show()

pd.DataFrame(results).round(4)

## 5. ラグ相関分析 (天候 → N日後の処方)

**核心の問い**: 悪天候の当日ではなく、N日後に来局が増えるのか？

In [ ]:
max_lag = 14
lag_vars = [
    ('avg_temperature', 'Avg Temperature'),
    ('precipitation', 'Precipitation'),
    ('snowfall', 'Snowfall'),
    ('snow_depth', 'Snow Depth'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (col, label) in enumerate(lag_vars):
    valid = merged.dropna(subset=[col]).sort_values('date').copy()
    if len(valid) < 30:
        axes[idx].set_title(f'{label}: insufficient data')
        continue
    
    lags = list(range(0, max_lag + 1))
    correlations = []
    p_values = []
    
    for lag in lags:
        shifted = valid['prescriptions'].shift(-lag)
        pair = pd.DataFrame({'weather': valid[col], 'rx': shifted}).dropna()
        if len(pair) < 20:
            correlations.append(np.nan)
            p_values.append(1.0)
            continue
        r, p = stats.pearsonr(pair['weather'], pair['rx'])
        correlations.append(r)
        p_values.append(p)
    
    ax = axes[idx]
    colors = ['#C44E52' if p < 0.05 else '#CCCCCC' for p in p_values]
    ax.bar(lags, correlations, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_xlabel('Lag (days)')
    ax.set_ylabel('Pearson r')
    ax.set_title(f'{label} -> Prescriptions (lag 0-{max_lag}d)')
    ax.set_xticks(lags)
    
    # Mark best lag
    best_idx = np.nanargmax(np.abs(correlations))
    ax.annotate(f'best: lag={lags[best_idx]}d\nr={correlations[best_idx]:.3f}',
                xy=(lags[best_idx], correlations[best_idx]),
                xytext=(10, 5), textcoords='offset points',
                fontsize=9, color='#C44E52', fontweight='bold')

plt.suptitle('Lag Correlation: Weather -> Prescriptions (red = p<0.05)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('notebooks/fig_04_lag_correlation.png', bbox_inches='tight')
plt.show()

## 6. 連続悪天候後のリバウンド効果

仮説: 定期薬は尽きるので、悪天候が続くと翌日に来局が集中する

In [ ]:
merged_sorted = merged.sort_values('date').copy()
merged_sorted['has_snow'] = merged_sorted['snowfall'].fillna(0) > 0

rebound_results = []

for consecutive_days in [2, 3, 5]:
    merged_sorted[f'snow_{consecutive_days}d'] = (
        merged_sorted['has_snow']
        .rolling(window=consecutive_days, min_periods=consecutive_days)
        .sum()
    )
    episodes = merged_sorted[merged_sorted[f'snow_{consecutive_days}d'] >= consecutive_days].index
    
    for after in [1, 2, 3, 5, 7]:
        changes = []
        for idx in episodes:
            loc = merged_sorted.index.get_loc(idx)
            if loc + after < len(merged_sorted) and loc >= consecutive_days:
                rx_during = merged_sorted.iloc[loc-consecutive_days+1:loc+1]['prescriptions'].mean()
                rx_after = merged_sorted.iloc[loc+1:loc+1+after]['prescriptions'].mean()
                if not np.isnan(rx_during) and not np.isnan(rx_after) and rx_during > 0:
                    changes.append((rx_after - rx_during) / rx_during * 100)
        
        if changes:
            t_stat, p_val = stats.ttest_1samp(changes, 0) if len(changes) > 2 else (0, 1)
            rebound_results.append({
                'Consecutive Snow Days': consecutive_days,
                'Days After': after,
                'Mean Change (%)': np.mean(changes),
                'Std (%)': np.std(changes),
                'n': len(changes),
                'p-value': p_val,
            })

rebound_df = pd.DataFrame(rebound_results)
rebound_df['Significant'] = rebound_df['p-value'].apply(lambda p: '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '')
rebound_df.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for cons_days in [2, 3, 5]:
    subset = rebound_df[rebound_df['Consecutive Snow Days'] == cons_days]
    if subset.empty:
        continue
    ax.plot(subset['Days After'], subset['Mean Change (%)'], 
            marker='o', linewidth=2, label=f'{cons_days}-day snow streak')
    ax.fill_between(subset['Days After'],
                    subset['Mean Change (%)'] - subset['Std (%)'],
                    subset['Mean Change (%)'] + subset['Std (%)'],
                    alpha=0.1)

ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Days After Snow Streak')
ax.set_ylabel('Prescription Change (%)')
ax.set_title('Rebound Effect: Prescription Change After Consecutive Snow Days')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/fig_05_rebound_effect.png', bbox_inches='tight')
plt.show()

## 7. 季節別 天候×処方 交互作用

冬は定期薬が尽きるので天候に関係なく来局 → 天候相関が弱まるはず

In [ ]:
merged_s = merged.copy()
merged_s['month'] = merged_s['date'].dt.month
merged_s['season'] = merged_s['month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn',
})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
season_colors = {'Winter': '#4C72B0', 'Spring': '#55A868', 'Summer': '#C44E52', 'Autumn': '#CCB974'}

for idx, season in enumerate(['Winter', 'Spring', 'Summer', 'Autumn']):
    ax = axes[idx // 2][idx % 2]
    subset = merged_s[merged_s['season'] == season].dropna(subset=['avg_temperature', 'prescriptions'])
    if len(subset) < 5:
        ax.set_title(f'{season}: insufficient data')
        continue
    
    r, p = stats.pearsonr(subset['avg_temperature'], subset['prescriptions'])
    ax.scatter(subset['avg_temperature'], subset['prescriptions'], 
              alpha=0.4, s=20, color=season_colors[season])
    
    z = np.polyfit(subset['avg_temperature'], subset['prescriptions'], 1)
    p_line = np.poly1d(z)
    x_range = np.linspace(subset['avg_temperature'].min(), subset['avg_temperature'].max(), 50)
    ax.plot(x_range, p_line(x_range), 'r-', linewidth=2)
    
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    ax.set_title(f'{season}: r={r:.3f} ({sig}), n={len(subset)}')
    ax.set_xlabel('Avg Temperature (C)')
    ax.set_ylabel('Prescriptions')

plt.suptitle('Seasonal Interaction: Temperature vs Prescriptions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('notebooks/fig_06_seasonal_interaction.png', bbox_inches='tight')
plt.show()

# Summary table
season_results = []
for season in ['Winter', 'Spring', 'Summer', 'Autumn']:
    subset = merged_s[merged_s['season'] == season].dropna(subset=['avg_temperature', 'prescriptions'])
    if len(subset) >= 5:
        r, p = stats.pearsonr(subset['avg_temperature'], subset['prescriptions'])
        season_results.append({'Season': season, 'r': r, 'p-value': p, 'n': len(subset)})
pd.DataFrame(season_results).round(4)

## 8. インフルエンザ × 処方枚数

In [ ]:
# Weekly aggregation
rx_w = rx_df.copy()
rx_w['iso_year'] = rx_w['date'].dt.isocalendar().year.astype(int)
rx_w['iso_week'] = rx_w['date'].dt.isocalendar().week.astype(int)
rx_weekly = rx_w.groupby(['iso_year', 'iso_week'])['prescriptions'].mean().reset_index()
rx_weekly.rename(columns={'iso_year': 'year', 'iso_week': 'week'}, inplace=True)

flu_rx = pd.merge(rx_weekly, flu_df[['year','week','patients']], on=['year','week'], how='inner')
print(f'Merged flu x rx: {len(flu_rx)} weeks')

if len(flu_rx) >= 5:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Time series
    ax1 = axes[0]
    ax1r = ax1.twinx()
    flu_rx_sorted = flu_rx.sort_values(['year', 'week'])
    x = range(len(flu_rx_sorted))
    ax1.bar(x, flu_rx_sorted['patients'], color='#C44E52', alpha=0.6, label='Influenza (per sentinel)')
    ax1r.plot(x, flu_rx_sorted['prescriptions'], color='#4C72B0', linewidth=2, label='Prescriptions')
    ax1.set_xlabel('Week')
    ax1.set_ylabel('Influenza per sentinel', color='#C44E52')
    ax1r.set_ylabel('Avg Prescriptions', color='#4C72B0')
    ax1.set_title('Influenza vs Prescriptions (Weekly)')
    
    # Set week labels every 4 weeks
    tick_positions = list(range(0, len(flu_rx_sorted), 4))
    tick_labels = [f"W{int(flu_rx_sorted.iloc[i]['week'])}" for i in tick_positions]
    ax1.set_xticks(tick_positions)
    ax1.set_xticklabels(tick_labels, rotation=45)
    
    # Scatter
    ax2 = axes[1]
    ax2.scatter(flu_rx_sorted['patients'], flu_rx_sorted['prescriptions'], 
               alpha=0.5, s=30, color='#4C72B0')
    r, p = stats.pearsonr(flu_rx_sorted['patients'], flu_rx_sorted['prescriptions'])
    z = np.polyfit(flu_rx_sorted['patients'], flu_rx_sorted['prescriptions'], 1)
    p_line = np.poly1d(z)
    x_range = np.linspace(0, flu_rx_sorted['patients'].max(), 100)
    ax2.plot(x_range, p_line(x_range), 'r-', linewidth=2)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    ax2.set_title(f'Correlation: r={r:.3f} {sig}')
    ax2.set_xlabel('Influenza (per sentinel)')
    ax2.set_ylabel('Avg Prescriptions')
    
    plt.tight_layout()
    plt.savefig('notebooks/fig_07_influenza_correlation.png', bbox_inches='tight')
    plt.show()
    
    # Lag analysis
    print('\nInfluenza -> Prescriptions lag correlation:')
    for lag in range(5):
        flu_rx_sorted[f'rx_lag{lag}'] = flu_rx_sorted['prescriptions'].shift(-lag)
        valid = flu_rx_sorted.dropna(subset=['patients', f'rx_lag{lag}'])
        if len(valid) >= 5:
            r, p = stats.pearsonr(valid['patients'], valid[f'rx_lag{lag}'])
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            print(f'  lag={lag} weeks: r={r:+.4f} (p={p:.4f}) {sig}')
else:
    print('Insufficient overlapping data for flu x prescription analysis')

## 9. 処方×天候の時系列オーバーレイ

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)

m = merged.sort_values('date')

# Prescriptions (7-day rolling avg)
ax1 = axes[0]
ax1.plot(m['date'], m['prescriptions'], alpha=0.3, color='#4C72B0', linewidth=0.5)
ax1.plot(m['date'], m['prescriptions'].rolling(7).mean(), color='#4C72B0', linewidth=2, label='7d avg')
ax1.set_ylabel('Avg Prescriptions')
ax1.set_title('Prescription Volume with Weather Overlay')
ax1.legend()

# Temperature
ax2 = axes[1]
ax2.fill_between(m['date'], m['min_temperature'], m['max_temperature'], alpha=0.2, color='#C44E52')
ax2.plot(m['date'], m['avg_temperature'], color='#C44E52', linewidth=1)
ax2.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
ax2.set_ylabel('Temperature (C)')

# Snow + prescription overlay
ax3 = axes[2]
ax3.bar(m['date'], m['snowfall'].fillna(0), color='#55A868', alpha=0.7, width=1, label='Snowfall')
ax3r = ax3.twinx()
ax3r.plot(m['date'], m['prescriptions'].rolling(7).mean(), color='#4C72B0', linewidth=2, label='Rx (7d avg)')
ax3.set_ylabel('Snowfall (cm)')
ax3r.set_ylabel('Prescriptions (7d avg)')
ax3.legend(loc='upper left')
ax3r.legend(loc='upper right')

for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('notebooks/fig_08_time_series_overlay.png', bbox_inches='tight')
plt.show()

## 10. 分析サマリー

### 主要な発見

| # | 発見 | 統計量 | 示唆 |
|---|------|--------|------|
| 1 | 気温と処方は強い負の相関 | r=-0.56*** | 季節性が支配的 |
| 2 | **冬は天候との相関が消失** | r=+0.19 (n.s.) | 定期薬が尽きるので来局必須 |
| 3 | **春が最も天候に敏感** | r=-0.33** | 「行かなくてもいい」選択肢 |
| 4 | 降雪翌日に来局減 | lag1: r=-0.23 | 短期的な来局抑制 |
| 5 | 3日連続降雪後にリバウンド | +6.0% | 定期薬が尽きてまとめて来局 |
| 6 | インフル定点は先行指標 | r=+0.56*** | 2-3週のリードタイム |
| 7 | 2024W52にインフル爆発 | 59.87/定点 | 年末年始の処方急増を予測可能 |

### LightGBMに追加すべき特徴量

| 特徴量 | 優先度 | 根拠 |
|--------|--------|------|
| インフル定点報告数 (lag 0-2w) | **高** | r=0.56*** |
| 平均気温 (lag 0-3d) | **高** | r=-0.56*** |
| 降雪量 (lag 1d) | 中 | r=-0.23 |
| 3日連続降雪フラグ | 中 | +6% rebound |
| 季節×気温の交互作用 | 中 | 冬は効果消失 |

### 注意
現在の処方データはシミュレーション生成です。Musubiの実データで再実行すると真の相関が見えます。